In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import numpy as np
from tqdm import tqdm

import matplotlib.pyplot as plt
import pickle
from dotenv import load_dotenv

load_dotenv() # load GEMINI_API_KEY

client = genai.Client()

from utils.prompting import extract_target_object_regex, create_inpaint_prompts, extract_target_object_gemini

In [ ]:
with open('data/bridge_fractal_train_n200.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
# TODO: Make Bulk/Batch Calls
trajectories = data['trajectories']
for traj in tqdm(trajectories, desc="Processing trajectories"):
    if traj.get('language_target_object'):
        continue
    robot_instruction = traj['language']
    # print(f"robot_instruction: {robot_instruction}")
    target_object = extract_target_object_gemini(client, robot_instruction=robot_instruction)
    # print(target_object)
    traj['language_target_object'] = target_object

In [ ]:
# Inpaint the image by removing the target object(s): {traj['language_target_object']}
i = 0
for traj in tqdm(trajectories, desc="Processing trajectories"):
    results = f"Traj_id: {traj['trajectory_id']}, dataset: {traj['dataset']}, Instr: {traj['language']}, Target Object: {traj['language_target_object']}"
    print(results)
    i += 1
    if i >= 5:
        break

In [ ]:
def inpaint_image_with_gemini(client, image_array, language_instruction, inpaint_prompt=None):
    """
    Inpaint an image by removing the target object using Gemini Image Edit API.
    
    :param image_array: numpy array of the image (256x256x3)
    :param language_instruction: language instruction (e.g., "pour almonds in pot")
    :param client: Gemini client
    :returns: Inpainted image as numpy array, same size as input
    """
    # Convert numpy array to PIL Image
    if image_array.dtype != np.uint8:
        image_array = (image_array * 255).astype(np.uint8)
    
    pil_image = Image.fromarray(image_array)
    
    # Create the inpainting prompt
    if not inpaint_prompt:
        base_prompt = "Inpaint the image by removing the target object(s) in the instruction."
        inpaint_prompt = f"{base_prompt}: '{language_instruction}'"
        
    try:
        # Make API call to Gemini
        response = client.models.generate_content(
            model="gemini-2.5-flash-image-preview",
            contents=[inpaint_prompt, pil_image],
        )
        
        # Extract the inpainted image from response
        for part in response.candidates[0].content.parts:
            if part.inline_data is not None:
                inpainted_pil = Image.open(BytesIO(part.inline_data.data))
                # Ensure the output image is same size as input (256x256)
                if inpainted_pil.size != (256, 256):
                    inpainted_pil = inpainted_pil.resize((256, 256), Image.LANCZOS)
                
                # Convert back to numpy array
                inpainted_array = np.array(inpainted_pil)
                return inpainted_array
        
        # If no image found in response, return original image
        print(f"Warning: No inpainted image returned for instruction: {language_instruction}")
        return None
        
    except Exception as e:
        print(f"Error inpainting image for instruction '{language_instruction}': {e}")
        return None


In [ ]:
import textwrap

def create_removal_inpainting_gemini(trajectories, client, num_samples=5e6, output_pkl_path=None, metadata=None, plot=False):
    """
    Test the inpainting method on a few sample trajectories.
    
    :param data: Dataset dictionary with trajectories
    :param client: Gemini client
    :param num_samples: Number of trajectories to test
    :returns: List of test results
    """    
    def calculate_num_samples_without_inpaintings(trajectories):
        return sum([1 for traj in trajectories if not all(traj.get(key) is not None for key in ['first_image_inpainted', 'last_image_inpainted'])])
    
    total_trajectories = calculate_num_samples_without_inpaintings(trajectories)
    total_images = total_trajectories * 2  # first + last image per trajectory
    
    print(f"Starting inpainting process for {total_trajectories} trajectories ({total_images} images total)")
    print(f"This may take a while depending on API response times...")
                        
    processed_images = 0
        
    for traj in tqdm(trajectories):
        if all(traj.get(key) is not None for key in ['first_image_inpainted', 'last_image_inpainted']):        # Test inpainting on first image
            continue

        target_obj = traj['language_target_object']
        print(f"\n--- Sample --- Dataset: {traj['dataset']}, ID: {traj['trajectory_id']}")
        print(f"Language instruction: {traj['language']}, target object: {target_obj}")

        inpaint_prompt = (
            f"Inpaint the image by removing the object(s): '{target_obj}'. "
            f"Do not alter the robotic arm or any background elements that are not the target object."
        )
        # inpaint_prompt = f"""Carefully remove ONLY the {target_obj} from this robotic manipulation scene. 

        #     REMOVE: {target_obj}

        #     PRESERVE EXACTLY:
        #     - The robotic arm/gripper (do not modify, move, or alter it in any way)
        #     - The table surface and background
        #     - All other objects not specified for removal
        #     - Natural lighting and shadows

        #     Fill the area where the {target_obj} was located with appropriate background (table surface, empty space, etc.) that matches the surrounding context. The result should look natural as if the {target_obj} was never there."""

        print("Inpainting first image...")
        first_inpainted = inpaint_image_with_gemini(
            client,
            traj['first_image'], 
            traj['language'], 
            inpaint_prompt=textwrap.dedent(inpaint_prompt)
        )
        
        # Test inpainting on last image
        print("Inpainting last image...")
        last_inpainted = inpaint_image_with_gemini(
            client,
            traj['last_image'], 
            traj['language'], 
            inpaint_prompt=inpaint_prompt
        )
        
        # Store results
        traj['first_image_inpainted'] = first_inpainted
        traj['last_image_inpainted'] = last_inpainted
        
        if plot:
            # Plot comparison for this sample
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            fig.suptitle(f"{traj['dataset']} ID:{traj['trajectory_id']}\\n'{traj['language']}'", 
                        fontsize=14, fontweight='bold')
            
            axes[0, 0].imshow(traj['first_image'])
            axes[0, 0].set_title('First Image (Original)')
            axes[0, 0].axis('off')
            
            axes[0, 1].imshow(first_inpainted)
            axes[0, 1].set_title('First Image (Inpainted)')
            axes[0, 1].axis('off')
            
            axes[1, 0].imshow(traj['last_image'])
            axes[1, 0].set_title('Last Image (Original)')
            axes[1, 0].axis('off')
            
            axes[1, 1].imshow(last_inpainted)
            axes[1, 1].set_title('Last Image (Inpainted)')
            axes[1, 1].axis('off')
            
            plt.tight_layout()
            plt.show()

        processed_images += 2

        # Save intermediate results every 20 trajectories
        if output_pkl_path and ((processed_images + 1) // 2) % 20 == 0:
            dataset = {
                'metadata': metadata,
                'trajectories': trajectories
            }
            with open(output_pkl_path, 'wb') as f:
                pickle.dump(dataset, f)
            print(f"  Saved intermediate results to {output_pkl_path}")
        
        if (processed_images // 2) >= num_samples:
            break   
    
    # Save final results
    if output_pkl_path:
        dataset = {
                'metadata': metadata,
                'trajectories': trajectories
            }
        with open(output_pkl_path, 'wb') as f:
            pickle.dump(dataset, f)
        print(f"Final dataset saved to: {output_pkl_path}")

    # Final summary
    print(f"\\n=== Inpainting Complete ===")
    print(f"Total trajectories processed: {total_trajectories}")
    print(f"Total images processed: {processed_images}/{total_images}")

    return trajectories

In [ ]:
# Test inpainting on 5 sample trajectories
output_path = 'data/bridge_fractal_train_n200_inpainted.pkl'
results = create_removal_inpainting_gemini(trajectories, client, num_samples=5e6, output_pkl_path=output_path, metadata=data['metadata'], plot=False)

In [ ]:
# data['metadata']['split'] = 'val'
# data['metadata']['created_through'] = 'gemini-2.5-flash-image-preview'
# data['metadata']['format_version'] = 1.2
# data['metadata']['created_for'] = 'ood_eval'
data['metadata']

In [ ]:
dataset = {
        'metadata': data['metadata'],
        'trajectories': trajectories
    }
with open(output_path, 'wb') as f:
    pickle.dump(dataset, f)

# results = create_removal_inpainting_gemini(trajectories, client, num_samples=5e6, output_pkl_path=output_path, metadata=data['metadata'], plot=False)

In [ ]:
for i, traj in enumerate(trajectories):
    if all(traj.get(key) is not None for key in ['first_image_inpainted', 'last_image_inpainted']):
        continue
    print(i)

In [ ]:
# idx = 87
# traj = trajectories[idx]

# target_obj = traj['language_target_object']
# inpaint_prompt = (
#     f"Inpaint the image by removing everything that looks like one of the following object(s): '{target_obj}'. "
#     f"Do not alter the robotic arm"
# )
# Image.fromarray(traj['last_image'])

In [ ]:
# # Test inpainting on last image
# print("Inpainting last image...")
# last_inpainted = inpaint_image_with_gemini(
#     client,
#     traj['last_image'], 
#     traj['language'], 
#     inpaint_prompt=inpaint_prompt
# )
# # Store results
# trajectories[idx]['last_image_inpainted'] = last_inpainted


#### Plot Inpainting Results

In [ ]:
from create_traj_viewer import create_interactive_viewer

# load inpainted dataset bridge_fractal_train_n200_inpainted.pkl
with open("data/bridge_fractal_train_n200_inpainted.pkl", "rb") as f:
    inpainted_dataset = pickle.load(f)

In [ ]:
# import pickle
# import h5py
# import json

# def pkl_to_hdf5(pkl_path, hdf5_path):
#     """Convert pickle file to HDF5."""
#     print(f"Loading {pkl_path}...")
#     with open(pkl_path, 'rb') as f:
#         data = pickle.load(f)
    
#     print(f"Converting to {hdf5_path}...")
#     with h5py.File(hdf5_path, 'w') as hf:
#         # Save metadata as JSON attribute
#         if 'metadata' in data:
#             hf.attrs['metadata'] = json.dumps(data['metadata'])
        
#         # Create trajectories group
#         traj_group = hf.create_group('trajectories')
        
#         trajectories = data['trajectories']
#         for i, traj in enumerate(trajectories):
#             traj_grp = traj_group.create_group(f'traj_{i}')
            
#             # Save numpy arrays as datasets
#             for key, value in traj.items():
#                 if hasattr(value, 'shape'):  # numpy array
#                     traj_grp.create_dataset(key, data=value)
#                 else:  # scalar values
#                     traj_grp.attrs[key] = value
            
#             if (i + 1) % 50 == 0:
#                 print(f"Processed {i + 1}/{len(trajectories)} trajectories")
    
#     print(f"Done! Saved {len(trajectories)} trajectories to {hdf5_path}")

# pkl_to_hdf5('data/bridge_fractal_train_n200_inpainted.pkl', 'data/bridge_fractal_train_n200_inpainted.hdf5')

In [ ]:
! pip list

In [ ]:
# Create interactive viewer with a subset of trajectories for testing
# Use subset_size to limit the number of trajectories (useful for large datasets)
viewer_path = create_interactive_viewer(
    inpainted_dataset, 
    output_path="trajectory_viewer_bridge_fractal_train_n200_inpainted.html",
    template_path="trajectory_viewer_template.html",
    subset_size=20  # Start with 20 trajectories for testing
)